# YOLOv7 Weed Detection Training
This notebook handles the dataset preparation and training for **YOLOv7**.

### ⚠️ Windows Stability Fixes
This version includes:
1. **Global Patcher**: Fixes `torch.load` security blocks.
2. **Cache Cleaner**: Deletes corrupted files before Training/Validation.
3. **Worker Fix**: Forces `--batch-size 8` during testing to prevent 'DataLoader worker' crashes on Windows.

In [1]:
import os
import sys
import json
import shutil
import random
import subprocess
import re
from pathlib import Path
import pandas as pd

# Settings
DATASET_ROOT = r"c:\Users\ahmad\Desktop\computer vision\cottonweed"
WORK_DIR = r"c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark"
FRACTION = 0.10  
EPOCHS = 50
BATCH_SIZE = 8
IMG_SIZE = 640
DEVICE = 0

CLASS_NAMES = ["weed1", "weed2", "weed3", "weed4", "weed5", "weed6", "weed7", "weed8", "weed9", "weed10", "weed11", "weed12"]

os.makedirs(WORK_DIR, exist_ok=True)

### 1. Setup YOLOv7 Repo & Global Patcher

In [2]:
repo_dir = Path(WORK_DIR) / "repos" / "yolov7"
if not repo_dir.exists():
    !git clone https://github.com/WongKinYiu/yolov7 "{repo_dir}"
    !pip install -r "{repo_dir}/requirements.txt"
else:
    print("YOLOv7 repo already exists.")

def global_patch_yolov7(root_path):
    """Forcefully adds weights_only=False to EVERY torch.load call in the entire repo folder."""
    print(f"Searching for torch.load calls in {root_path}...")
    count = 0
    for p in Path(root_path).rglob("*.py"):
        with open(p, 'r', encoding='utf-8') as f:
            content = f.read()
        
        if 'torch.load(' in content:
            new_content = re.sub(
                r'torch\.load\(([^)]+)\)', 
                lambda m: f"torch.load({m.group(1)}, weights_only=False)" if 'weights_only' not in m.group(0) else m.group(0),
                content
            )
            
            if new_content != content:
                with open(p, 'w', encoding='utf-8') as f:
                    f.write(new_content)
                print(f"  [PATCHED] {p.relative_to(root_path)}")
                count += 1
    print(f"Successfully patched {count} files.")

global_patch_yolov7(repo_dir)

def clear_broken_cache(subset_path):
    """Deletes .cache files to prevent 'unpickling stack underflow' errors."""
    print(f"Cleaning up dataset cache files in {subset_path}...")
    for cache_file in subset_path.rglob("*.cache"):
        try:
            os.remove(cache_file)
            print(f"  - Removed {cache_file.name}")
        except Exception as e:
            print(f"  - Could not remove {cache_file.name}: {e}")

# Pretrained weights
weights_path = Path(WORK_DIR) / "weights" / "yolov7.pt"
if not weights_path.exists():
    os.makedirs(weights_path.parent, exist_ok=True)
    !curl -L https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt -o "{weights_path}"

YOLOv7 repo already exists.
Searching for torch.load calls in c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\repos\yolov7...
Successfully patched 0 files.


### 2. Prepare Dataset Subset

In [3]:
def create_subset(fraction):
    subset_name = f"subset_{int(fraction * 100)}"
    subset_root = Path(WORK_DIR) / "datasets" / subset_name
    
    if (subset_root / "SUBSET_READY.json").exists():
        return subset_root

    print(f"Creating {subset_name} subset...")
    os.makedirs(subset_root / "images" / "train", exist_ok=True)
    os.makedirs(subset_root / "labels" / "train", exist_ok=True)
    
    for split in ["val", "test"]:
        src_img = Path(DATASET_ROOT) / "images" / split
        if src_img.exists():
            shutil.copytree(src_img, subset_root / "images" / split, dirs_exist_ok=True)
            shutil.copytree(Path(DATASET_ROOT) / "labels" / split, subset_root / "labels" / split, dirs_exist_ok=True)

    train_images = list((Path(DATASET_ROOT) / "images" / "train").glob("*.jpg"))
    random.seed(42)
    selected = random.sample(train_images, int(len(train_images) * fraction))
    
    for img_path in selected:
        shutil.copy2(img_path, subset_root / "images" / "train")
        lbl_path = Path(DATASET_ROOT) / "labels" / "train" / f"{img_path.stem}.txt"
        if lbl_path.exists():
            shutil.copy2(lbl_path, subset_root / "labels" / "train")

    yaml_content = f"train: { (subset_root / 'images' / 'train').as_posix() }\nval: { (subset_root / 'images' / 'val').as_posix() }\nnc: {len(CLASS_NAMES)}\nnames: {CLASS_NAMES}"
    with open(subset_root / "data.yaml", "w") as f:
        f.write(yaml_content)
    
    with open(subset_root / "SUBSET_READY.json", "w") as f:
        json.dump({"fraction": fraction}, f)
    return subset_root

subset_path = create_subset(FRACTION)

### 3. Training (Real-time output)

In [4]:
import subprocess
import sys

exp_name = f"yolov7_data{int(FRACTION*100)}_aug"
exp_dir = Path(WORK_DIR) / "runs" / exp_name
data_yaml = subset_path / "data.yaml"
best_ckpt = exp_dir / "weights" / "best.pt"

if best_ckpt.exists():
    print("Training already finished. Skipping to validation.")
else:
    clear_broken_cache(subset_path)
    
    cmd = [
        sys.executable, str(repo_dir / "train.py"),
        "--workers", "0",
        "--device", str(DEVICE),
        "--batch-size", str(BATCH_SIZE),
        "--data", str(data_yaml),
        "--img", str(IMG_SIZE), str(IMG_SIZE),
        "--cfg", str(repo_dir / "cfg/training/yolov7.yaml"),
        "--weights", str(weights_path),
        "--name", exp_name,
        "--project", str(Path(WORK_DIR) / "runs"),
        "--hyp", str(repo_dir / "data/hyp.scratch.p5.yaml"),
        "--epochs", str(EPOCHS)
    ]
    
    print(f"Starting training: {' '.join(cmd)}\n")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=str(repo_dir))
    for line in process.stdout:
        print(line, end="")
    process.wait()

Training already finished. Skipping to validation.


### 4. Validation & Results

In [5]:
clear_broken_cache(subset_path)

test_cmd = [
    sys.executable, str(repo_dir / "test.py"),
    "--weights", str(exp_dir / "weights" / "best.pt"),
    "--data", str(data_yaml),
    "--img", str(IMG_SIZE),
    "--batch-size", "8"  # Reduced from default 32 to prevent memory crashes
]
print(f"Starting test: {' '.join(test_cmd)}\n")
process = subprocess.Popen(test_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=str(repo_dir))
for line in process.stdout:
    print(line, end="")
process.wait()

# Parse results.txt
res_txt = exp_dir / "results.txt"
if res_txt.exists():
    with open(res_txt, "r") as f:
        lines = f.readlines()
        if lines:
            last_line = lines[-1].split()
            summary = {
                "model": "yolov7",
                "fraction": FRACTION,
                "map50": float(last_line[-2]),
                "precision": float(last_line[-4]),
                "recall": float(last_line[-3])
            }
            with open(exp_dir / "EXPERIMENT_DONE.json", "w") as f:
                json.dump(summary, f)

Cleaning up dataset cache files in c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\datasets\subset_10...
  - Removed val.cache
Starting test: e:\miniconda3\envs\whisper\python.exe c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\repos\yolov7\test.py --weights c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\runs\yolov7_data10_aug\weights\best.pt --data c:\Users\ahmad\Desktop\computer vision\weed_yolo_benchmark\datasets\subset_10\data.yaml --img 640 --batch-size 8

Namespace(weights=['c:\\Users\\ahmad\\Desktop\\computer vision\\weed_yolo_benchmark\\runs\\yolov7_data10_aug\\weights\\best.pt'], data='c:\\Users\\ahmad\\Desktop\\computer vision\\weed_yolo_benchmark\\datasets\\subset_10\\data.yaml', batch_size=8, img_size=640, conf_thres=0.001, iou_thres=0.65, task='val', device='', single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project='runs/test', name='exp', exist_ok=False, no_trace=False,